In [26]:
# IMPORT LIBRARIES & PERSIAPAN DATASET UTUH (TANPA PREPROCESS.IPYNB)

import os
import pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import AdamW, SGD, RMSprop, Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import class_weight

print("\n[INFO] Libraries berhasil dimuat!")


[INFO] Libraries berhasil dimuat!


In [27]:
# --- KONFIGURASI GLOBAL ---
DATASET_DIR = pathlib.Path("data_cropped") # Pastikan foldernya benar
IMG_SIZE = 224 # Menggantikan img_height
BATCH_SIZE = 32 # Menggantikan batch_size
LEARNING_RATE = 0.0005
EPOCHS = 75
K_FOLDS = 7
DROPOUT_RATE = 0.3
OPTIMIZER_NAME = 'Adam'

# --- LOAD PATH DATASET ---
all_image_paths = list(DATASET_DIR.glob('*/*.png'))
all_image_paths = [str(path) for path in all_image_paths]

class_names_list = sorted([item.name for item in DATASET_DIR.glob('*') if item.is_dir()])
label_to_index = dict((name, index) for index, name in enumerate(class_names_list))
all_image_labels = [label_to_index[pathlib.Path(path).parent.name] for path in all_image_paths]

X = np.array(all_image_paths)
y = np.array(all_image_labels)

print(f"[INFO] Total dataset siap K-Fold: {len(X)} gambar")

# --- FUNGSI PIPELINE DATA (MASKING & AUGMENTASI) ---
def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.io.decode_image(img, channels=4, expand_animations=False)
    img = tf.cast(img, tf.float32)
    # Masking
    rgb = img[..., :3]
    alpha = img[..., 3:4] / 255.0
    img_fixed = rgb * alpha
    # Resize & Normalisasi
    img_resized = tf.image.resize(img_fixed, [IMG_SIZE, IMG_SIZE])
    img_final = img_resized / 255.0
    img_final.set_shape([IMG_SIZE, IMG_SIZE, 3])
    
    label_one_hot = tf.one_hot(label, depth=len(class_names_list))
    return img_final, label_one_hot

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1)
])

def augment(image, label):
    return data_augmentation(image, training=True), label

[INFO] Total dataset siap K-Fold: 904 gambar


In [28]:
# MEMBANGUN ARSITEKTUR CUSTOM CNN 

def build_aflatoxin_uv_cnn():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="Input_Layer")
    
    x = layers.Conv2D(16, (3, 3), activation='relu', padding='same', name="Conv1")(inputs)
    x = layers.BatchNormalization(name="BatchNorm1")(x) # <- Tambahan agar model cepat fokus
    x = layers.MaxPooling2D((2, 2), name="Pool1")(x)
    
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same', name="Conv2")(x)
    x = layers.BatchNormalization(name="BatchNorm2")(x) # <- memastikan sinyal fitur pendaran UV tidak meredup atau meledak (vanishing/exploding gradients) saat masuk ke lapisan yang lebih dalam
    x = layers.MaxPooling2D((2, 2), name="Pool2")(x)
    
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same', name="Conv3")(x)
    x = layers.BatchNormalization(name="BatchNorm3")(x)
    x = layers.MaxPooling2D((2, 2), name="Pool3")(x)
    
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same', name="Conv4")(x)
    x = layers.BatchNormalization(name="BatchNorm4")(x)
    x = layers.MaxPooling2D((2, 2), name="Pool4")(x)
    
    # --- DUAL POOLING (Mendeteksi Kecerahan Tertinggi & Luas Area) ---
    max_pool = layers.GlobalMaxPooling2D(name="Peak_Brightness_Detector")(x) # <- Bertugas mencari titik pendaran paling terang (Intensitas puncak racun).
    avg_pool = layers.GlobalAveragePooling2D(name="Spatial_Extent_Detector")(x) # <- Bertugas menghitung luas sebaran pendaran secara keseluruhan di permukaan jagung.
    merged = layers.Concatenate(name="Feature_Fusion")([max_pool, avg_pool]) # <- memberikan gambaran yang lebih lengkap tentang pola pendaran UV pada jagung: seberapa terang racunnya, dan seberapa luas serangannya.
    
    # --- KEPALA KLASIFIKASI DENGAN OTAK DIPERBESAR ---
    # 2. Naikkan neuron dari 128 ke 256 agar bisa menyimpan pola rumit
    x = layers.Dense(256, activation='relu', name="Dense_Hidden_1")(merged)
    x = layers.Dropout(DROPOUT_RATE)(x)
    # 2. Hakim Penengah / Sistem Corong (Merangkum 256 laporan menjadi 128 intisari)
    x = layers.Dense(128, activation='relu', name="Dense_Hidden_2")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(4, activation='softmax', name="Output_Classification")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Custom_CNN_Dual_Pooling")
    return model

# Print summary sekali saja di awal
dummy_model = build_aflatoxin_uv_cnn()
print("="*60)
print("ARSITEKTUR MODEL BERHASIL DIBANGUN")
print("="*60)
dummy_model.summary()

ARSITEKTUR MODEL BERHASIL DIBANGUN


Model: "Custom_CNN_Dual_Pooling"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 224, 224,  │        448 │ Input_Layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ BatchNorm1          │ (None, 224, 224,  │         64 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Pool1               │ (None, 112, 112,  │          0 │ BatchNorm1[0][0]  │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv2 (Conv2D)      │ (None, 112, 112,  │      4,640 │ Pool1[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ BatchNorm2          │ (None, 112, 112,  │        128 │ Conv2[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Pool2               │ (None, 56, 56,    │          0 │ BatchNorm2[0][0]  │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv3 (Conv2D)      │ (None, 56, 56,    │     18,496 │ Pool2[0][0]       │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ BatchNorm3          │ (None, 56, 56,    │        256 │ Conv3[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Pool3               │ (None, 28, 28,    │          0 │ BatchNorm3[0][0]  │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv4 (Conv2D)      │ (None, 28, 28,    │     73,856 │ Pool3[0][0]       │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ BatchNorm4          │ (None, 28, 28,    │        512 │ Conv4[0][0]       │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Pool4               │ (None, 14, 14,    │          0 │ BatchNorm4[0][0]  │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Peak_Brightness_De… │ (None, 128)       │          0 │ Pool4[0][0]       │
│ (GlobalMaxPooling2… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Spatial_Extent_Det… │ (None, 128)       │          0 │ Pool4[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Feature_Fusion      │ (None, 256)       │          0 │ Peak_Brightness_… │
│ (Concatenate)       │                   │            │ Spatial_Extent_D… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_Hidden_1      │ (None, 256)       │     65,792 │ Feature_Fusion[0

 Total params: 197,604 (771.89 KB)

 Trainable params: 197,124 (770.02 KB)

 Non-trainable params: 480 (1.88 KB)

In [29]:
# MULAI PERULANGAN K-FOLD (BAGIAN 4, 5, 6, 7, 8 MASUK KE DALAM SINI)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
global_accuracies = []

for fold_no, (train_index, val_index) in enumerate(skf.split(X, y), 1):
    print("\n" + "★"*60)
    print(f"🚀 MEMULAI TRAINING FOLD KE-{fold_no}")
    print("★"*60)
    
    # --- A. SETUP DATA UNTUK FOLD INI ---
    X_train_paths, X_val_paths = X[train_index], X[val_index]
    y_train_labels, y_val_labels = y[train_index], y[val_index]
    
    # ==============================================================================
    # --- TAMBAHAN BARU: OVERSAMPLING ON-THE-FLY (KHUSUS DATA LATIH) ---
    # ==============================================================================
    # Asumsi: y adalah label One-Hot Encoding (misal Kelas 4 adalah indeks ke-3)
    # 1. Cari indeks array yang merupakan Kelas 4
    class_4_indices = np.where(y_train_labels == 3)[0]
    
    # 2. Ekstrak path dan label untuk Kelas 4 tersebut
    X_train_class_4 = X_train_paths[class_4_indices]
    y_train_class_4 = y_train_labels[class_4_indices]
    
    # 3. Kloning/Duplikasi dengan menggabungkan ke data asli (Menjadi tepat 2x lipat)
    X_train_paths = np.concatenate([X_train_paths, X_train_class_4])
    y_train_labels = np.concatenate([y_train_labels, y_train_class_4])
    
    # 4. Acak (Shuffle) array agar data kloningan tidak menumpuk di akhir antrean
    shuffle_idx = np.random.permutation(len(X_train_paths))
    X_train_paths = X_train_paths[shuffle_idx]
    y_train_labels = y_train_labels[shuffle_idx]
    # ==============================================================================

    # Hitung Bobot Kelas Otomatis untuk Fold ini
    # Karena y_train_labels berupa One-Hot, kita ubah ke 1D (integer) agar sklearn tidak error
    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)
    fold_class_weights = dict(enumerate(weights))
    
    # Buat Dataset
    train_ds = tf.data.Dataset.from_tensor_slices((X_train_paths, y_train_labels))
# ... (LANJUTKAN KE KODE ASLI ANDA MULAI DARI SINI: train_ds = train_ds.map(process_path, ...))
    train_ds = train_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.shuffle(len(X_train_paths)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    val_ds = tf.data.Dataset.from_tensor_slices((X_val_paths, y_val_labels))
    val_ds = val_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    # Aliasing val_ds menjadi test_ds agar kode evaluasi Anda di bawah tidak error
    test_ds = val_ds 
    
    # --- B. RESET & COMPILE MODEL (Cell 4 Lama) ---
    tf.keras.backend.clear_session()
    model = build_aflatoxin_uv_cnn()
    
    # lr_schedule = CosineDecay(initial_learning_rate=0.0005, #lr paling stabil
    #                           decay_steps=EPOCHS * (504 // BATCH_SIZE))
    if OPTIMIZER_NAME.lower() == 'adam':
        opt = Adam(learning_rate=LEARNING_RATE)
    elif OPTIMIZER_NAME.lower() == 'sgd':
        opt = SGD(learning_rate=LEARNING_RATE, momentum=0.9)
    elif OPTIMIZER_NAME.lower() == 'adamw':
        opt = AdamW(learning_rate=LEARNING_RATE)
    else:
        opt = RMSprop(learning_rate=LEARNING_RATE)

    model.compile(optimizer=opt, loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), metrics=['accuracy'])

    # Simpan model dengan nama berbeda tiap fold
    current_model_path = f'best_custom_cnn_uv_fold{fold_no}.keras'
    callbacks_list = [
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
        ModelCheckpoint(current_model_path, monitor='val_accuracy', save_best_only=True, verbose=0)
    ]

    # --- C. TRAINING (Cell 5 Lama) ---
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks_list,
        class_weight=fold_class_weights, 
        verbose=1
    )

    # =================================================================================================
    # 6. EVALUASI AKHIR PADA DATA TEST (KODE ASLI ANDA, TIDAK DIUBAH LOGIKANYA)
    # =================================================================================================
    print(f"\n[INFO] Melakukan Evaluasi Fold {fold_no}...")

    try:
        model.load_weights(current_model_path)
        print(f"✓ Model terbaik berhasil dimuat dari {current_model_path}")
    except:
        print("⚠️ Menggunakan bobot model epoch terakhir")

    Y_pred_probs = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(Y_pred_probs, axis=1) 

    y_true_onehot = np.concatenate([y for x, y in test_ds], axis=0)
    y_true = np.argmax(y_true_onehot, axis=1)

    class_names = ['Kelas 1 (1 PPB)', 'Kelas 2 (2 PPB)', 'Kelas 3 (3 PPB)', 'Kelas 4 (4 PPB)']

    test_acc = accuracy_score(y_true, y_pred)
    global_accuracies.append(test_acc * 100)
    
    confidence_scores = np.max(Y_pred_probs, axis=1)
    avg_conf = np.mean(confidence_scores)

    report = classification_report(y_true, y_pred, target_names=class_names, digits=4, output_dict=True)
    print("\n" + "=" * 60)
    print(f"📋 CLASSIFICATION REPORT FOLD {fold_no}")
    print("=" * 60)
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    cm_str = str(cm.tolist())

    plot_folder = "history_plots_customCNN" 
    if not os.path.exists(plot_folder):
        os.makedirs(plot_folder)

    current_time_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    # Nama file disesuaikan agar fold tidak tertimpa
    plot_filename = f"{current_time_str}_Fold{fold_no}_CM_Acc{test_acc*100:.2f}.png" 
    plot_filepath = os.path.join(plot_folder, plot_filename)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, annot_kws={"size": 14})
    plt.title(f'Confusion Matrix Fold {fold_no} (Acc: {test_acc*100:.2f}%)')
    plt.ylabel('Actual Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()

    plt.savefig(plot_filepath, dpi=300, bbox_inches='tight')
    plt.close() # Ditutup agar tidak menumpuk di layar

    print(f"✅ Akurasi Fold {fold_no}: {test_acc*100:.2f}%")

    # =================================================================================================
    # 7. VISUALISASI HASIL PELATIHAN (KODE ASLI ANDA)
    # =================================================================================================
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    plt.title(f'Kurva Akurasi Fold {fold_no} (Acc: {test_acc*100:.2f}%)', fontsize=14)
    plt.xlabel('Epoch')
    plt.ylabel('Akurasi')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss (Error)', linewidth=2)
    plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    plt.title(f'Kurva Loss Fold {fold_no}', fontsize=14)
    plt.xlabel('Epoch')
    plt.ylabel('Loss (Categorical Crossentropy)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()

    folder_name = "visualizations_CUSTOM"
    if not os.path.exists(folder_name):
        os.makedirs(folder_name) 
    
    file_name = f"{current_time_str}_Fold{fold_no}_Acc{test_acc*100:.2f}.png"
    save_path = os.path.join(folder_name, file_name)

    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    # =================================================================================================
    # 8. SISTEM PENYIMPANAN LOG (KODE ASLI ANDA)
    # =================================================================================================
    LOG_FILE_PATH = "experiment_log_custom_cnn.csv"

    log_data = {
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Fold_No": fold_no, # Tambahan untuk tracking fold
        "Model": "Custom CNN (Dual-Pooling UV)",
        "Total_Params": model.count_params(),
        "Learning_Rate": LEARNING_RATE,
        "Batch_Size": BATCH_SIZE, 
        "Dropout_Rate": DROPOUT_RATE,
        "Img_Size": IMG_SIZE,  
        "Use_Class_Weights": "Yes (Dynamic K-Fold)",
        "Accuracy": round(test_acc, 4),
        "Avg_Confidence": round(avg_conf, 4),
        "Confusion_Matrix": cm_str
    }

    for idx, label in enumerate(class_names):
        metrics = report[label] 
        log_data[f"C{idx+1}_Prec"] = round(metrics['precision'], 4)
        log_data[f"C{idx+1}_Rec"] = round(metrics['recall'], 4)
        log_data[f"C{idx+1}_F1"] = round(metrics['f1-score'], 4)

    df_new_log = pd.DataFrame([log_data])

    if not os.path.exists(LOG_FILE_PATH):
        df_new_log.to_csv(LOG_FILE_PATH, index=False)
    else:
        df_new_log.to_csv(LOG_FILE_PATH, mode='a', header=False, index=False)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🚀 MEMULAI TRAINING FOLD KE-1
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
Epoch 1/75
27/27 ━━━━━━━━━━━━━━━━━━━━ 14s 417ms/step - accuracy: 0.2688 - loss: 4.2613 - val_accuracy: 0.2692 - val_loss: 1.3824 - learning_rate: 5.0000e-04
Epoch 2/75
27/27 ━━━━━━━━━━━━━━━━━━━━ 12s 411ms/step - accuracy: 0.3016 - loss: 1.8080 - val_accuracy: 0.2692 - val_loss: 1.3818 - learning_rate: 5.0000e-04
Epoch 3/75
27/27 ━━━━━━━━━━━━━━━━━━━━ 12s 412ms/step - accuracy: 0.3415 - loss: 1.4724 - val_accuracy: 0.2692 - val_loss: 1.3779 - learning_rate: 5.0000e-04
Epoch 4/75
27/27 ━━━━━━━━━━━━━━━━━━━━ 12s 410ms/step - accuracy: 0.3791 - loss: 1.2903 - val_accuracy: 0.2692 - val_loss: 1.3727 - learning_rate: 5.0000e-04
Epoch 5/75
27/27 ━━━━━━━━━━━━━━━━━━━━ 12s 407ms/step - accuracy: 0.4131 - loss: 1.2650 - val_accuracy: 0.2692 - val_loss: 1.3815 - learning_rate: 5.0000e-04
Epoch 6/75
27/27 ━━━━━━━━━━━━━━━━━━━━ 12s 403ms/step - accuracy

In [8]:
# # MULAI PERULANGAN K-FOLD (BAGIAN 4, 5, 6, 7, 8 MASUK KE DALAM SINI)

# skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
# global_accuracies = []

# for fold_no, (train_index, val_index) in enumerate(skf.split(X, y), 1):
#     print("\n" + "★"*60)
#     print(f"🚀 MEMULAI TRAINING FOLD KE-{fold_no}")
#     print("★"*60)
    
#     # --- A. SETUP DATA UNTUK FOLD INI ---
#     X_train_paths, X_val_paths = X[train_index], X[val_index]
#     y_train_labels, y_val_labels = y[train_index], y[val_index]
    
#     # Hitung Bobot Kelas Otomatis untuk Fold ini (Menggantikan Cell 2 Lama)
#     weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)
#     fold_class_weights = dict(enumerate(weights))
    
#     # Buat Dataset
#     train_ds = tf.data.Dataset.from_tensor_slices((X_train_paths, y_train_labels))
#     train_ds = train_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
#     train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
#     train_ds = train_ds.shuffle(len(X_train_paths)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
#     val_ds = tf.data.Dataset.from_tensor_slices((X_val_paths, y_val_labels))
#     val_ds = val_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
#     val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
#     # Aliasing val_ds menjadi test_ds agar kode evaluasi Anda di bawah tidak error
#     test_ds = val_ds 
    
#     # --- B. RESET & COMPILE MODEL (Cell 4 Lama) ---
#     tf.keras.backend.clear_session()
#     model = build_aflatoxin_uv_cnn()
    
#     # lr_schedule = CosineDecay(initial_learning_rate=0.0005, #lr paling stabil
#     #                           decay_steps=EPOCHS * (504 // BATCH_SIZE))
#     if OPTIMIZER_NAME.lower() == 'adam':
#         opt = Adam(learning_rate=LEARNING_RATE)
#     elif OPTIMIZER_NAME.lower() == 'sgd':
#         opt = SGD(learning_rate=LEARNING_RATE, momentum=0.9)
#     elif OPTIMIZER_NAME.lower() == 'adamw':
#         opt = AdamW(learning_rate=LEARNING_RATE)
#     else:
#         opt = RMSprop(learning_rate=LEARNING_RATE)

#     # def apply_ordinal_label_smoothing(y_true_onehot):
#     #     """
#     #     Fungsi untuk mengubah label One-Hot menjadi Ordinal Label Smoothing.
#     #     Asumsi: y_true_onehot memiliki bentuk (batch_size, 4)
#     #     """
#     #     # Matriks OLS buatan kita (Baris = Kelas Asli, Kolom = Distribusi Probabilitas)
#     #     ols_matrix = tf.constant([
#     #         [0.90, 0.07, 0.02, 0.01], # Kelas 0 (1 PPB)
#     #         [0.04, 0.90, 0.04, 0.02], # Kelas 1 (2 PPB)
#     #         [0.02, 0.04, 0.90, 0.04], # Kelas 2 (3 PPB)
#     #         [0.01, 0.02, 0.07, 0.90]  # Kelas 3 (4 PPB)
#     #     ], dtype=tf.float32)
        
#     #     # Mencari indeks kelas aslinya (0, 1, 2, atau 3)
#     #     true_class_indices = tf.argmax(y_true_onehot, axis=1)
        
#     #     # Mengambil baris matriks OLS yang sesuai dengan kelas aslinya
#     #     y_smoothed = tf.gather(ols_matrix, true_class_indices)
        
#     #     return y_smoothed

#     model.compile(optimizer=opt, loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), metrics=['accuracy'])

#     # 2. SEBELUM data masuk ke model.fit, ubah dulu label di Dataset Anda.
#     # Jika Anda menggunakan tf.data.Dataset:
#     # train_ds_ols = train_ds.map(lambda x, y: (x, apply_ordinal_label_smoothing(y)))
#     # val_ds_ols = val_ds.map(lambda x, y: (x, apply_ordinal_label_smoothing(y)))

#     # Simpan model dengan nama berbeda tiap fold
#     current_model_path = f'best_custom_cnn_uv_fold{fold_no}.keras'
#     callbacks_list = [
#         EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
#         ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
#         ModelCheckpoint(current_model_path, monitor='val_accuracy', save_best_only=True, verbose=0)
#     ]

#     # --- C. TRAINING (Cell 5 Lama) ---
#     history = model.fit(
#         # train_ds_ols,
#         # validation_data=val_ds_ols,
#         train_ds,
#         validation_data=val_ds,
#         epochs=EPOCHS,
#         callbacks=callbacks_list,
#         class_weight=fold_class_weights, 
#         verbose=1
#     )

#     # =================================================================================================
#     # 6. EVALUASI AKHIR PADA DATA TEST (KODE ASLI ANDA, TIDAK DIUBAH LOGIKANYA)
#     # =================================================================================================
#     print(f"\n[INFO] Melakukan Evaluasi Fold {fold_no}...")

#     try:
#         model.load_weights(current_model_path)
#         print(f"✓ Model terbaik berhasil dimuat dari {current_model_path}")
#     except:
#         print("⚠️ Menggunakan bobot model epoch terakhir")

#     Y_pred_probs = model.predict(test_ds, verbose=1)
#     y_pred = np.argmax(Y_pred_probs, axis=1) 

#     y_true_onehot = np.concatenate([y for x, y in test_ds], axis=0)
#     y_true = np.argmax(y_true_onehot, axis=1)

#     class_names = ['Kelas 1 (1 PPB)', 'Kelas 2 (2 PPB)', 'Kelas 3 (3 PPB)', 'Kelas 4 (4 PPB)']

#     test_acc = accuracy_score(y_true, y_pred)
#     global_accuracies.append(test_acc * 100)
    
#     confidence_scores = np.max(Y_pred_probs, axis=1)
#     avg_conf = np.mean(confidence_scores)

#     report = classification_report(y_true, y_pred, target_names=class_names, digits=4, output_dict=True)
#     print("\n" + "=" * 60)
#     print(f"📋 CLASSIFICATION REPORT FOLD {fold_no}")
#     print("=" * 60)
#     print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

#     cm = confusion_matrix(y_true, y_pred)
#     cm_str = str(cm.tolist())

#     plot_folder = "history_plots_customCNN" 
#     if not os.path.exists(plot_folder):
#         os.makedirs(plot_folder)

#     current_time_str = datetime.now().strftime("%Y%m%d_%H%M%S")
#     # Nama file disesuaikan agar fold tidak tertimpa
#     plot_filename = f"{current_time_str}_Fold{fold_no}_CM_Acc{test_acc*100:.2f}.png" 
#     plot_filepath = os.path.join(plot_folder, plot_filename)

#     plt.figure(figsize=(8, 6))
#     sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
#                 xticklabels=class_names, yticklabels=class_names, annot_kws={"size": 14})
#     plt.title(f'Confusion Matrix Fold {fold_no} (Acc: {test_acc*100:.2f}%)')
#     plt.ylabel('Actual Label', fontsize=12)
#     plt.xlabel('Predicted Label', fontsize=12)
#     plt.tight_layout()

#     plt.savefig(plot_filepath, dpi=300, bbox_inches='tight')
#     plt.close() # Ditutup agar tidak menumpuk di layar

#     print(f"✅ Akurasi Fold {fold_no}: {test_acc*100:.2f}%")

#     # =================================================================================================
#     # 7. VISUALISASI HASIL PELATIHAN (KODE ASLI ANDA)
#     # =================================================================================================
#     plt.figure(figsize=(14, 5))

#     plt.subplot(1, 2, 1)
#     plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
#     plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
#     plt.title(f'Kurva Akurasi Fold {fold_no} (Acc: {test_acc*100:.2f}%)', fontsize=14)
#     plt.xlabel('Epoch')
#     plt.ylabel('Akurasi')
#     plt.legend()
#     plt.grid(True, linestyle='--', alpha=0.7)

#     plt.subplot(1, 2, 2)
#     plt.plot(history.history['loss'], label='Train Loss (Error)', linewidth=2)
#     plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
#     plt.title(f'Kurva Loss Fold {fold_no}', fontsize=14)
#     plt.xlabel('Epoch')
#     plt.ylabel('Loss (Categorical Crossentropy)')
#     plt.legend()
#     plt.grid(True, linestyle='--', alpha=0.7)

#     plt.tight_layout()

#     folder_name = "visualizations_CUSTOM"
#     if not os.path.exists(folder_name):
#         os.makedirs(folder_name) 
    
#     file_name = f"{current_time_str}_Fold{fold_no}_Acc{test_acc*100:.2f}.png"
#     save_path = os.path.join(folder_name, file_name)

#     plt.savefig(save_path, dpi=300, bbox_inches='tight')
#     plt.close()

#     # =================================================================================================
#     # 8. SISTEM PENYIMPANAN LOG (KODE ASLI ANDA)
#     # =================================================================================================
#     LOG_FILE_PATH = "experiment_log_custom_cnn.csv"

#     log_data = {
#         "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
#         "Fold_No": fold_no, # Tambahan untuk tracking fold
#         "Model": "Custom CNN (Dual-Pooling UV)",
#         "Total_Params": model.count_params(),
#         "Learning_Rate": LEARNING_RATE,
#         "Batch_Size": BATCH_SIZE, 
#         "Dropout_Rate": DROPOUT_RATE,
#         "Img_Size": IMG_SIZE,  
#         "Use_Class_Weights": "Yes (Dynamic K-Fold)",
#         "Accuracy": round(test_acc, 4),
#         "Avg_Confidence": round(avg_conf, 4),
#         "Confusion_Matrix": cm_str
#     }

#     for idx, label in enumerate(class_names):
#         metrics = report[label] 
#         log_data[f"C{idx+1}_Prec"] = round(metrics['precision'], 4)
#         log_data[f"C{idx+1}_Rec"] = round(metrics['recall'], 4)
#         log_data[f"C{idx+1}_F1"] = round(metrics['f1-score'], 4)

#     df_new_log = pd.DataFrame([log_data])

#     if not os.path.exists(LOG_FILE_PATH):
#         df_new_log.to_csv(LOG_FILE_PATH, index=False)
#     else:
#         df_new_log.to_csv(LOG_FILE_PATH, mode='a', header=False, index=False)

In [30]:
# --- REKAPITULASI AKHIR K-FOLD ---
print("\n" + "="*60)
print("🏆 KESIMPULAN HASIL 5-FOLD CROSS VALIDATION")
print("="*60)
for i, acc in enumerate(global_accuracies):
    print(f"Fold {i+1}: {acc:.2f}%")
print("-" * 30)
print(f"RATA-RATA AKURASI KESELURUHAN: {np.mean(global_accuracies):.2f}% (+/- {np.std(global_accuracies):.2f}%)")
print("="*60)


🏆 KESIMPULAN HASIL 5-FOLD CROSS VALIDATION
Fold 1: 63.85%
Fold 2: 62.02%
Fold 3: 66.67%
Fold 4: 64.34%
Fold 5: 63.57%
Fold 6: 64.34%
Fold 7: 58.91%
------------------------------
RATA-RATA AKURASI KESELURUHAN: 63.38% (+/- 2.23%)
